# Observation-only conditional report

This small notebook uses **untrained** weights to demonstrate the API, not a recovery or calibration claim. Install `pymc-mfm[diagnostics]` and a Python notebook kernel. Replace the model with `from_pretrained("local-artifact")` for trained weights. No generator, truth, tracking server or network access is required.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import torch
from IPython.display import Image, display

from pymc_mfm import MFM, ModelConfig, decompose
from pymc_mfm.diagnostics import build_report, render_report

torch.manual_seed(12)
model = MFM(
    ModelConfig(
        d_model=8,
        n_layers=1,
        n_heads=2,
        d_ff=16,
        flow_hidden=16,
        flow_steps_sample=2,
        n_controls=2,
        n_demand=1,
    )
).eval()
week = np.arange(12)
observations = {
    "spend": np.column_stack([100 + week * 5, 80 + week * 2]),
    "controls": np.column_stack([np.sin(week), np.cos(week)]),
    "context_outcome": 500 + week[:8] * 4,
}
result = decompose(
    model,
    observations,
    n_samples=16,
    channel_names=("Search", "Social"),
    control_names=("Price", "Holiday"),
    units="sales units",
    provenance={"data_split": "observation_only"},
)
result.components.shape, result.graphs.shape

In [ ]:
output = TemporaryDirectory(prefix="mfm-report-")
root = Path(output.name)
manifest = build_report(result, root / "report")
manifest["provenance"], manifest["statuses"]

In [ ]:
replayed = render_report(root / "report", root / "replayed")
# Rendering consumes only saved evidence; it does not call the model.
for artifact in replayed["artifacts"]:
    if artifact["kind"] == "figure":
        display(Image(filename=str(root / "replayed" / artifact["path"])))

Truth-dependent recovery, coverage and calibration panels are explicitly unavailable. Component totals exclude observation noise. Fixed-graph output describes a supplied hypothesis; empirical graph grouping is not a calibrated conditional uncertainty estimate. Context and query calls are separate marginals.